# GP02B Six Development Cases

Inspect the six-case development batch and repair traces instrumentation.

This source copy comes from **gp02b-v2-7.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

# GP02B v2.7 — Autonomous Tool Selection, Six Development Tasks, Automatic HTML Traces

Based on GP02B v2.6. **Only the mandatory investigation-tool prerequisite is rolled back** in the agent guidance, tool wrapper, and the result integrity checks. Gemma remains free to choose when and whether to use `read_file`, `get_code_neighbors`, or other retrieval tools. The nine existing callable schemas remain available. Preserve the original Google starter guidance and the v2.6 prompt/context and `edit_file` clarifications.

The harness still protects test files, records tool usage, retains independent qualification/verification, and enforces the existing model, retrieval, call and time limits. All six selected tasks, sequential execution, automatic HTML creation, per-task ZIP export and prior evidence are unchanged. GP02B v2.7 uses a distinct register and prompt fingerprint; v2.4–v2.6 evidence remains preserved.


## GP02B 01 — Restore frozen v2.4 evidence (standalone)

Loads the verified results archive and reconstructs the original helper sources, prompt hashes, sampling limits, and minimal baseline metadata. **Does not require GP02A in-memory variables or a running server.**

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import dataclasses, importlib.util, importlib, hashlib, json, sys, ast, copy

KAGGLE_BASE = Path('/kaggle/working/Gemma4_Paper')
BASELINE_IDS = ('20261106T212242Z_8f5325d2', '20261107T012921Z_8c546a07')
BASELINE_ZIPS = [KAGGLE_BASE/f'GP02A_{rid}'/'GP02A_results.zip' for rid in BASELINE_IDS]
BASELINE_ZIPS += list(Path('/kaggle/input').rglob('GP02A_results.zip'))
BASELINE_ZIP = next((p for p in BASELINE_ZIPS if p.is_file()), None)
if BASELINE_ZIP is None:
    raise FileNotFoundError('Attach the verified GP02A v2.4 results ZIP to this notebook.')

PROMPT_ORDER = ('system_custom_base.md', 'investigation_protocol.md', 'tools_code_graph.md',
                'tools_code_retrieval.md', 'tools_file_operations.md', 'tools_testing.md', 'tools_execution.md')
with ZipFile(BASELINE_ZIP) as z:
    original = json.loads(z.read('00_summary.json'))
    SETTINGS_V24 = json.loads(z.read('settings.json'))
    BASELINE_TRIAL_CONFIG = json.loads(z.read('trial_config.json'))
    ARCHIVED_SERVER_CONFIG = json.loads(z.read('server_config.json'))
    assert original['status'] == 'PATCH_VERIFIED' and original['version'] == 'GP02A_v2.4'
    assert SETTINGS_V24['max_output_tokens'] == 2048 and SETTINGS_V24['agent_time_minutes'] == 15.0
    SOURCE_V24 = {Path(x).name: z.read(x).decode('utf-8') for x in z.namelist()
                  if x.startswith('run_sources/') and x.count('/') == 1 and x.endswith('.py')}
    PROMPTS_V24 = {key: z.read(f'run_sources/submission/prompts/{key}').decode('utf-8')
                   for key in PROMPT_ORDER}
    SAMPLING_V24 = z.read('run_sources/submission/configs/sampling.yaml').decode('utf-8')
assert len(SOURCE_V24) == 12, f'Unexpected v2.4 helper set: {sorted(SOURCE_V24)}'
composed = '\n\n'.join('# COMPONENT: ' + x + '\n' + PROMPTS_V24[x].strip() for x in PROMPT_ORDER) + '\n'
PROMPT_SHA = hashlib.sha256(composed.encode()).hexdigest()
assert PROMPT_SHA == '3bcd0d47c330e110ffb92c18f912e0c7e3776d0d94fb76a6ce7d3c0bba0466ea'
assert PROMPT_SHA == SETTINGS_V24['expected_composed_system_sha256']

# v2.6: two narrowly scoped agent-facing prompt corrections only.
# Preserve the original v2.4 prompt/hash above as historical evidence.
PROMPTS_V26 = dict(PROMPTS_V24)
_orig_file_guide = PROMPTS_V26['tools_file_operations.md']
_readline = 'The development harness may permit up to 500 lines and 27,500 characters; this is a ceiling, not a request to consume all of it.'
assert _orig_file_guide.count(_readline) == 1
PROMPTS_V26['tools_file_operations.md'] = _orig_file_guide.replace(
    _readline,
    'The current development harness permits at most 150 lines and 8,000 characters per read_file result. Start with short targeted ranges (e.g., 20–80 lines), and request more only if needed. Every returned character uses conversation context.'
)
_editline = 'edit_file(filepath, old_string, new_string, allow_multiple=False): preferred for existing files. Include unique surrounding context. Inspect the returned diff and detect unintended replacements.'
assert PROMPTS_V26['tools_file_operations.md'].count(_editline) == 1
PROMPTS_V26['tools_file_operations.md'] = PROMPTS_V26['tools_file_operations.md'].replace(
    _editline,
    """edit_file(filepath, old_string, new_string, allow_multiple=False): preferred for existing files. REQUIRED separate tool arguments: filepath (string), old_string (string with exact current text), and new_string (string with replacement text). allow_multiple is optional and defaults to False.
VALID separate-argument example: {"filepath": "pkg/module.py", "old_string": "original_text", "new_string": "replacement_text"}.
Do NOT omit old_string. Do NOT embed the literal text `,old_string:` or any other tool argument inside new_string; that is a malformed call, not a valid separate argument. If you do not know exact old_string, call read_file for the relevant lines before editing. Use a short, unambiguous matching block with unique surrounding context. Inspect the returned diff and detect unintended replacements. If a call is rejected for a missing required field, repair the separate argument object before retrying."""
)
_orig_test_guide = PROMPTS_V26['tools_testing.md']
_cmdline = 'run_command(command): execute focused tests, inspection commands and short Python probes in /workspace. Avoid bare pytest, pytest ., unittest discovery and unrelated sweeps. Do not run pip install or network calls.'
assert _orig_test_guide.count(_cmdline) == 1
PROMPTS_V26['tools_testing.md'] = _orig_test_guide.replace(
    _cmdline,
    """run_command(command): execute focused tests, inspection commands and short Python probes in /workspace. Avoid bare pytest, pytest ., unittest discovery and unrelated sweeps. Do not run pip install or network calls.
The command tool returns at most 6,000 characters per stdout/stderr stream. Keep returned output smaller when possible: target relevant files and tests, abbreviate very long matching lines, and avoid dumping large fixture files or terminal-rendering strings. `head -n 20` alone does NOT limit characters within a line; use an appropriate per-line character bound for inspection commands. Preserve meaningful error messages and test outcomes rather than suppressing failures. All returned tool text consumes the model's cumulative context; avoid repetitive retrieval."""
)

_compose_v26 = '\n\n'.join('# COMPONENT: ' + key + '\n' + PROMPTS_V26[key].strip() for key in PROMPT_ORDER) + '\n'
PROMPT_SHA_V26 = hashlib.sha256(_compose_v26.encode('utf-8')).hexdigest()
assert PROMPT_SHA_V26 != PROMPT_SHA
assert {key for key in PROMPT_ORDER if PROMPTS_V26[key] != PROMPTS_V24[key]} == {
    'tools_file_operations.md', 'tools_testing.md'
}
print('v2.6 amended prompt SHA256:', PROMPT_SHA_V26)
print('Prompt amendments limited to two usage-guide files; v2.4 frozen prompt preserved.')


# v2.7: make investigation tool selection autonomous; do not force tool calls.
# Keep every v2.6 fix above unchanged and preserve the original archived prompts.
PROMPTS_V27 = dict(PROMPTS_V26)
_src = PROMPTS_V27['system_custom_base.md']
_before = '2. DIAGNOSE with evidence. Complete the mandatory investigation protocol before changing implementation files. Clearly distinguish observed facts from assumptions.'
_after = '2. DIAGNOSE with evidence. Select the investigation tools and sequence that best fit the issue and available information. Clearly distinguish observed facts from assumptions.'
assert _src.count(_before) == 1
_src = _src.replace(_before, _after)
_before = 'MUST call read_file and get_code_neighbors successfully before modifying code, with arguments YOU select from the public issue and observed source. A graph error is a blocked investigation, not an exemption.'
_after = 'Choose whichever available tools and arguments are useful for the task. Neither read_file nor get_code_neighbors is a mandatory prerequisite to editing, running commands, or submitting. A graph error is a tool limitation, not permission to invent evidence.'
assert _src.count(_before) == 1
PROMPTS_V27['system_custom_base.md'] = _src.replace(_before, _after)
PROMPTS_V27['investigation_protocol.md'] = '''## Investigation guidance — agent-selected
Understand the public issue, expected behaviours and likely affected source. Inspect relevant implementation, visible tests, execution errors, or the code graph as needed to support a repair.

Available tools include read_file, get_code_neighbors, get_code_subgraph, search_similar_code and run_command. You may select, omit, and order tools according to what the task actually requires. Do not make a tool call solely to satisfy a prerequisite. Use grounded symbols and paths; inspect returned evidence before relying on it. If a tool returns an error or no relevant evidence, use another appropriate approach and report important limitations rather than fabricating evidence.

When independent requests are useful, you may request multiple tools in the same model response where supported; do not invent dependent arguments before seeing their sources. Keep retrieval focused, conserve context and execution budget, and move from diagnosis to a minimal fix and meaningful tests when evidence is sufficient. Source edits and submission are not gated on successful calls to any specific investigation tool.
'''
PROMPTS_V27['tools_code_graph.md'] = '''## Code graph tools — optional investigation aids
get_code_neighbors(node, edge_type=None, max_neighbors=50) inspects local graph relationships for a real repository symbol identified from the issue or available source. When useful, prefer max_neighbors 20–30 to keep the response focused. The graph is OPTIONAL; neither an invocation nor a nonempty result is required before editing or submitting a repair.
get_code_subgraph(nodes) explores relationships among several already identified nodes. It returns an induced subgraph, not a complete repository graph.
A graph edge is not proof of runtime behaviour. Verify consequential claims against source or tests. If a symbol is missing or a tool fails, adjust your approach as appropriate; do not fabricate graph results or stop solely to satisfy a graph-tool rule. Gemma chooses which tools to use and all tool arguments.
'''
_compose_v27 = '\n\n'.join('# COMPONENT: ' + k + '\n' + PROMPTS_V27[k].strip() for k in PROMPT_ORDER) + '\n'
PROMPT_SHA_V27 = hashlib.sha256(_compose_v27.encode('utf-8')).hexdigest()
assert PROMPT_SHA_V27 not in (PROMPT_SHA_V26, PROMPT_SHA)
assert {k for k in PROMPT_ORDER if PROMPTS_V27[k] != PROMPTS_V26[k]} == {
    'system_custom_base.md', 'investigation_protocol.md', 'tools_code_graph.md'
}
assert PROMPTS_V27['tools_file_operations.md'] == PROMPTS_V26['tools_file_operations.md']
assert PROMPTS_V27['tools_testing.md'] == PROMPTS_V26['tools_testing.md']
assert not any('MUST call read_file' in PROMPTS_V27[k] or
               'mandatory gate' in PROMPTS_V27[k].lower() or
               'mandatory investigation' in PROMPTS_V27[k].lower()
               for k in PROMPT_ORDER)
print('v2.7 autonomous-tool prompt SHA256:', PROMPT_SHA_V27)
print('Previous v2.6 context and edit_file corrections: PRESERVED')
assert 'max_output_tokens: 2048' in SAMPLING_V24
COMPETITION = Path(SETTINGS_V24['competition_root'])
assert (COMPETITION/'tasks.jsonl').is_file(), 'Attach Kaggle competition input.'
assert hashlib.sha256((COMPETITION/'tasks.jsonl').read_bytes()).hexdigest() == BASELINE_TRIAL_CONFIG['tasks_sha256']

# These are the exact original offline wheel pins from the verified GP02A v2.4 setup.
CORE_PINS_V24 = [
    {'distribution':'adk-eval-core','version':'0.1.0','sha256':'e614e1b2fe8d886a634871e4c57f9887e1cc66e8c27300f8503a43c6e3ca82f3'},
    {'distribution':'adk-submission','version':'0.2.13','sha256':'f156f0bd84580ccb9334726d114d7a5481f16d3a6e2887cc4d276ced0ef8d99d'},
    {'distribution':'swegemma','version':'0.2.11','sha256':'9ad876581f3acde8ec3c369858f0834e2b9f41a3cffce3ca42f09cb090aea9d3'},
]
BASE_ASSETS = {
    'tasks_sha256': BASELINE_TRIAL_CONFIG['tasks_sha256'],
    'snapshot_sha256': BASELINE_TRIAL_CONFIG['snapshot_pins']['httpx_3672'],
    'verification_source_sha256': BASELINE_TRIAL_CONFIG['verification_source_sha256'],
    'agent_runner_source_sha256': BASELINE_TRIAL_CONFIG['agent_runner_source_sha256'],
    'core_pins': CORE_PINS_V24,
    'sources': SOURCE_V24,
    'protocols': PROMPTS_V24,
}
GP02B_ROOT = KAGGLE_BASE/'GP02B_Cross_Task'
GP02B_ROOT.mkdir(parents=True, exist_ok=True)
print('FROZEN V2.4 ARCHIVE: PASS')
print('Baseline:', BASELINE_ZIP)
print('System SHA256:', PROMPT_SHA)
print('Protected test passes:', original['candidate_junit']['passed'])
print('No prior GP02A Python variables required.')

## GP02B 01B — Install offline packages (CPU)

Uses the original GP02A v2.4 installation logic only if required modules are missing; no external downloads.

In [ ]:
# Uses the proven GP02A v2.4 offline installation path. No network access.
_ORIGINAL_CPU05 = 'import glob\nimport importlib\nimport logging\nimport os\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n# Configure environment variables for offline vLLM serving and LiteLLM routing\nos.environ[\'TRANSFORMERS_NO_TF\'] = \'1\'\nos.environ[\'TRANSFORMERS_NO_FLAX\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nos.environ[\'VLLM_NO_USAGE_STATS\'] = \'1\'\nos.environ[\'VLLM_WORKER_MULTIPROC_METHOD\'] = \'spawn\'\nos.environ[\'VLLM_ENGINE_READY_TIMEOUT_S\'] = \'1200\'\nos.environ[\'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS\'] = \'1\'\nos.environ[\'NCCL_DEBUG\'] = \'WARN\'\nos.environ[\'LITELLM_LOCAL_MODEL_COST_MAP\'] = \'True\'\nos.environ[\'LITELLM_TELEMETRY\'] = \'False\'\nos.environ[\'LITELLM_LOG\'] = \'WARNING\'\nos.environ[\'OPENAI_API_KEY\'] = os.environ.get(\'OPENAI_API_KEY\', \'EMPTY\')\nos.environ[\'LOCAL_API_KEY\'] = os.environ.get(\'LOCAL_API_KEY\', \'EMPTY\')\nos.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\nos.environ[\'OTEL_SDK_DISABLED\'] = \'true\'\nos.environ[\'OTEL_PYTHON_DISABLED\'] = \'true\'\n\n\ndef configure_logging() -> None:\n    """Silence verbose third-party loggers."""\n    for name in (\n        \'LiteLLM\',\n        \'LiteLLM Router\',\n        \'LiteLLM Proxy\',\n        \'litellm\',\n        \'httpx\',\n        \'httpcore\',\n        \'openai\',\n        \'urllib3\',\n        \'opentelemetry\',\n        \'opentelemetry.context\',\n    ):\n        lg = logging.getLogger(name)\n        lg.setLevel(logging.WARNING)\n        if \'litellm\' in name.lower():\n            lg.propagate = False\n            lg.handlers.clear()\n\n\nconfigure_logging()\n\nWHEELHOUSE_DIR = Path(\'/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse\')\n\n# Remove broken cutlass .pth hooks if present\nfor pth_pattern in (\n    \'/usr/local/lib/python*/dist-packages/*cutlass*.pth\',\n    \'/usr/local/lib/python*/site-packages/*cutlass*.pth\',\n):\n    for pth in glob.glob(pth_pattern):\n        try:\n            os.unlink(pth)\n        except OSError:\n            pass\n\n# Restore PEP 440 \'+cu128\' wheel filenames stripped by Kaggle dataset uploads\ntmp_whl = Path(\'/tmp/wheelhouse\')\ntmp_whl.mkdir(parents=True, exist_ok=True)\nfor w in WHEELHOUSE_DIR.glob(\'*.whl\'):\n    if \'cutlass\' in w.name.lower():\n        continue\n    target_name = (\n        w.name.replace(\'cu128\', \'+cu128\')\n        if (\'cu128\' in w.name and \'+\' not in w.name)\n        else w.name\n    )\n    target = tmp_whl / target_name\n    if not target.exists():\n        try:\n            os.symlink(w, target)\n        except OSError:\n            shutil.copyfile(w, target)\n\nwheels = sorted(str(w) for w in tmp_whl.glob(\'*.whl\'))\nprint(f\'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...\')\nsubprocess.run(\n    [sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--no-deps\', \'--force-reinstall\', *wheels],\n    check=True,\n)\nimportlib.invalidate_caches()\nconfigure_logging()\nprint(\'Wheelhouse installation complete.\')\n\n# CPU 05: verify installed distributions are importable in this exact kernel.\n# This is the mandatory gate before executing GPU model-startup code.\nimport importlib\n_required_imports = {\n    \'adk_submission\': \'adk-submission\',\n    \'swegemma\': \'swegemma\',\n    \'google.adk\': \'google-adk\',\n}\n_failed_imports = {}\nfor _module, _distribution in _required_imports.items():\n    try:\n        _pkg = importlib.import_module(_module)\n        print(f\'[PASS] {_module} (from {_pkg.__file__ if hasattr(_pkg, "__file__") else "namespace"})\', flush=True)\n    except Exception as _exc:\n        _failed_imports[_module] = f\'{type(_exc).__name__}: {_exc}\'\n        print(f\'[FAIL] {_module}: {_failed_imports[_module]}\', flush=True)\nif _failed_imports:\n    raise RuntimeError(\n        \'CPU 05 offline package installation did not establish a usable runtime: \'\n        + repr(_failed_imports)\n        + \'. Verify that the attached wheelhouse contains adk-submission and its dependencies, \'\n        + \'and that pip uses the notebook kernel interpreter (sys.executable).\'\n    )\nprint(\'CPU OFFLINE DEPENDENCY IMPORT GATE: PASS\', flush=True)\n'
# Fresh Kaggle kernel: ensure the exact official offline toolchain is importable.
import importlib, importlib.util
_REQUIRED = ('adk_submission', 'swegemma', 'google.adk')
def _is_importable(name):
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        return False
if all(_is_importable(k) for k in _REQUIRED):
    print('Offline agent packages already available; no reinstall needed.')
else:
    print('Installing pinned offline Kaggle wheelhouse packages.')
    exec(compile(_ORIGINAL_CPU05, '<frozen_v24_offline_install>', 'exec'), globals())
for _pkg in _REQUIRED:
    importlib.import_module(_pkg)
print('GP02B OFFLINE DEPENDENCY GATE: PASS')

## GP02B 01C — Restore archived starter submission (CPU)

Reconstructs exact original agent configuration, prompts, evaluation configuration and adapter files.

In [ ]:
# Reconstruct original v2.4 starter submission from evidence, not another notebook's variables.
AGENT_DIR = Path('/kaggle/working/GP02B_v24_starter_submission')
AGENT_DIR.mkdir(parents=True, exist_ok=True)
with ZipFile(BASELINE_ZIP) as z:
    entries = [x for x in z.namelist() if x.startswith('run_sources/submission/') and not x.endswith('/')]
    for member in entries:
        rel = member[len('run_sources/submission/'):]
        if not rel or '..' in Path(rel).parts:
            raise ValueError('Unsafe archived submission path: ' + member)
        target = AGENT_DIR/rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(member))
# GP02B FIX: archive run_sources/submission/ is the ALREADY PREPARED output.
# The v2.4 prepare() method expects the ORIGINAL starter registration and
# removes code_analyzer itself. Restore the archived originals as its INPUTS.
with ZipFile(BASELINE_ZIP) as z:
    original_filenames = {
        'google_original_agent.yaml': 'agent.yaml',
        'google_original_eval_config.yaml': 'eval_config.yaml',
        'prompts/google_original_system.md': 'prompts/system.md',
        'configs/google_original_sampling.yaml': 'configs/sampling.yaml',
    }
    for archived_rel, starter_rel in original_filenames.items():
        dest = AGENT_DIR/starter_rel
        dest.write_bytes(z.read('run_sources/submission/'+archived_rel))
assert (AGENT_DIR/'agent.yaml').is_file()
assert 'max_output_tokens: 16384' in (AGENT_DIR/'configs/sampling.yaml').read_text()
# These are the original starter inputs, NOT the final agent outputs.
# Stage 4 will remove the analyzer, recompose the exact v2.4 prompt,
# enforce max_output_tokens=2048, and verify its SHA-256 fingerprint.
assert (AGENT_DIR/'adapters/main_lora/adapter_model.safetensors').is_file()
print('Restored',len(entries),'frozen starter submission files to',AGENT_DIR)

## GP02B 01D — Initialise this notebook’s vLLM server (GPU)

One-time model load in a fresh session. Matches the original Gemma 4 31B L4×4 serving profile. Displays an elapsed-time heartbeat.

In [ ]:
# Start a NEW model service in this notebook/kernel when no service is already running.
# A restarted Kaggle session cannot reuse the previous Python server_instance.
import os, re, time, threading
from datetime import datetime, timezone
import requests, torch, litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')
MODEL_NAME = SETTINGS_V24['model_name']
MODEL_PATH = Path(SETTINGS_V24['model_path'])
assert MODEL_PATH.is_dir(), f'Attach original 31B model checkpoint: {MODEL_PATH}'
assert discover_declared_models(AGENT_DIR) == {MODEL_NAME} or MODEL_NAME in discover_declared_models(AGENT_DIR), 'Archived submission model differs from original.'
assert torch.cuda.is_available() and torch.cuda.device_count() == 4, 'Baseline requires four L4 GPUs (TP=4).'
assert all('L4' in torch.cuda.get_device_name(i) for i in range(4)), 'Expected original L4 x4 hardware.'
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters,'adapters',None) or {}
assert len(adapter_map) == 2, f'Unexpected v2.4 adapters: {list(adapter_map)}'
_v24_server = ARCHIVED_SERVER_CONFIG['config']
assert _v24_server['tensor_parallel_size'] == 4 and _v24_server['max_model_len'] == 32768

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16', gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True, enable_lora=True,
    max_loras=2, max_lora_rank=_v24_server['max_lora_rank'],
    scheduler_reserve_full_isl=False, tensor_parallel_size=4,
    startup_timeout=1200, extra_args=tuple(_v24_server['extra_args']),
)
# Avoid duplicate starts if the same kernel already owns a running server.
if globals().get('server_instance') is None:
    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
    _stop = threading.Event()
    _started = time.monotonic()
    def _heartbeat():
        while not _stop.wait(10):
            print(f'[vLLM] loading {MODEL_NAME} | elapsed={time.monotonic()-_started:.0f}s',flush=True)
    _thread = threading.Thread(target=_heartbeat,daemon=True)
    _thread.start()
    try:
        print('Starting fresh Gemma 31B vLLM service on L4 x4...',flush=True)
        server_instance.start()
    finally:
        _stop.set()
        _thread.join(timeout=2)
    print(f'Server READY in {time.monotonic()-_started:.1f}s at {server_instance.base_url}',flush=True)
else:
    print('Reusing server_instance from THIS GP02B kernel:',server_instance.base_url)
response = requests.get(server_instance.base_url.rstrip('/')+'/models',
                        headers={'Authorization':'Bearer '+INFERENCE_API_KEY},timeout=20)
response.raise_for_status()
assert response.json().get('data'), 'Server has no registered models.'
models = server_instance.create_model_registry(
    aliases=[MODEL_NAME], model_prefix='openai/',api_key=INFERENCE_API_KEY)
SETTINGS_V24['starter_submission_dir'] = str(AGENT_DIR)
SETTINGS_V24['model_path'] = str(MODEL_PATH)
print('GP02B SERVING GATE: PASS | original model, adapters and sampling profile')

## GP02B 02 — Freeze the task cohort and confirm snapshots (CPU only)

The initial cohort is four **previously shortlisted development tasks**. `requests_6757` remains on infrastructure hold and is not silently included. The source snapshot, repository and base commit are checked against the pinned published task list; task selection is frozen before inference. If a task does not qualify in this session, mark it `QUALIFICATION_HOLD` and do not invoke Gemma on it.

In [ ]:
# Public metadata only: never put patch/test_patch/hints/required test nodes into agent inputs.
FROZEN_TASK_MANIFEST = {
    'rich_2943':    {'repo': 'Textualize/rich', 'base_commit': '610fd75c433c3dfcd9c296e8b91363979127b9fa'},
    'rich_3130':    {'repo': 'Textualize/rich', 'base_commit': '720800e6930d85ad027b1e9bd0cbb96b5e994ce3'},
    'fastapi_5624': {'repo': 'fastapi/fastapi', 'base_commit': 'c6487ed632056e450d844846a1b63be551a3cbc6'},
    'fastapi_14479':{'repo': 'fastapi/fastapi', 'base_commit': '2b212ddd7604891d42e2ffbafe764c334230bdb7'},
}
TASK_ROWS = {}
with (COMPETITION/'tasks.jsonl').open(encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row['instance_id'] in FROZEN_TASK_MANIFEST:
            TASK_ROWS[row['instance_id']] = {k:row[k] for k in ('instance_id','repo','base_commit','problem_statement','created_at') if k in row}
assert set(TASK_ROWS) == set(FROZEN_TASK_MANIFEST)
PUBLIC_TASKS = {}
SNAPSHOT_PINS = {}
for task_id, expected in FROZEN_TASK_MANIFEST.items():
    row = TASK_ROWS[task_id]
    assert (row['repo'], row['base_commit']) == (expected['repo'], expected['base_commit']), task_id
    snapshot = COMPETITION/'snapshots'/f'{task_id}.tgz'
    assert snapshot.is_file(), f'Missing snapshot for {task_id}: {snapshot}'
    SNAPSHOT_PINS[task_id] = hashlib.sha256(snapshot.read_bytes()).hexdigest()
    PUBLIC_TASKS[task_id] = {key: row[key] for key in ('instance_id','repo','base_commit','problem_statement','created_at') if key in row}
    print(f'[PASS] {task_id:<15} {expected["repo"]:<22} snapshot {SNAPSHOT_PINS[task_id][:12]}')

FROZEN_COHORT = {'purpose':'development cross-task baseline, NOT held-out evaluation',
                 'agent_configuration':'frozen GP02A v2.4', 'system_sha256':PROMPT_SHA,
                 'tasks_sha256':BASE_ASSETS['tasks_sha256'],
                 'candidate_order':list(FROZEN_TASK_MANIFEST),
                 'tasks':{k:{**FROZEN_TASK_MANIFEST[k], 'snapshot_sha256':SNAPSHOT_PINS[k]}
                          for k in FROZEN_TASK_MANIFEST}}
cohort_path=GP02B_ROOT/'frozen_development_cohort.json'
if cohort_path.exists():
    assert json.loads(cohort_path.read_text()) == FROZEN_COHORT, 'Frozen task cohort changed across notebook runs.'
else:
    cohort_path.write_text(json.dumps(FROZEN_COHORT,indent=2))
print('Frozen GP02B task manifest:',cohort_path)

## GP02B 03 — CPU adapter: generalise only HTTPX task hardcodes

This is a strictly scoped modification to the *harness*, not the agent: replace fixed `httpx_3672` identifiers with the selected public task ID, snapshot pin and repository import check. Preserve the v2.4 prompt, tool gate, model generation settings and the independent evaluator unchanged. No task-specific prompt instructions are added. Validate every exact edit and compile all helper sources before any GPU call.

In [ ]:
def replace_exact(text, old, new, expected_count=1):
    actual=text.count(old)
    if actual!=expected_count:
        raise AssertionError(f'Adapter contract mismatch: expected {expected_count}, got {actual}: {old[:90]}')
    return text.replace(old,new)

def make_gp02b_sources(base):
    sources=dict(base)
    s=sources['workflow.py']
    s=replace_exact(s, '        self.settings, self.assets = settings, assets\n',
                      '        self.settings, self.assets = settings, assets\n        self.task_id = settings["task_id"]\n')
    s=replace_exact(s,"'snapshots/httpx_3672.tgz'", "'snapshots' / (self.task_id + '.tgz')")
    s=replace_exact(s,"('encode/httpx', '4acf5c2c37714cc63b5cf71b3e284fca83c90311')",
                     "(self.settings['task_expected_repo'], self.settings['task_expected_commit'])")
    s=replace_exact(s,"'evaluator_only/httpx_3672' / phase", "('evaluator_only/' + self.task_id) / phase")
    s=replace_exact(s,"'evaluator_only/httpx_3672/' + phase", "'evaluator_only/' + self.task_id + '/' + phase", 2)
    s=replace_exact(s,"'evaluator_only/httpx_3672'", "('evaluator_only/' + self.task_id)")
    s=s.replace("'httpx_3672'", 'self.task_id')
    assert 'httpx_3672' not in s
    sources['workflow.py']=s

    s=sources['smoke_worker.py']
    s=replace_exact(s,"'snapshots/httpx_3672.tgz'", "'snapshots' / (config['task_id'] + '.tgz')")
    s=s.replace("'httpx_3672'", "config['task_id']")
    s=replace_exact(s,"check_import_origin'](original_exec, sandbox_id, 'httpx',",
                     "check_import_origin'](original_exec, sandbox_id, task.repo_name.replace('-', '_'),")
    assert 'httpx_3672' not in s
    # Determine repository package from the pinned public manifest, not an undocumented Task attribute.
    s=replace_exact(s, "task.repo_name.replace('-', '_')",
      "config['task_expected_repo'].rsplit('/', 1)[-1].replace('-', '_')")
    sources['smoke_worker.py']=s

    s=sources['tool_preflight.py']
    s=replace_exact(s,"'httpx' in pair[0].lower()", "task.repo.split('/')[-1].lower() in pair[0].lower()")
    sources['tool_preflight.py']=s

    # Researcher-facing HTML only: accept the 5B stage marker, and select the
    # actual evaluated repository instead of displaying old HTTPX metrics.
    s=sources['review.py']
    s=replace_exact(s, "trial.artifacts / 'evaluator_only/httpx_3672' / name / 'result.json'",
                    "trial.artifacts / 'evaluator_only' / trial.settings['task_id'] / name / 'result.json'")
    s=replace_exact(s, "trial.artifacts / 'evaluator_only/httpx_3672/baseline/result.json'",
                    "trial.artifacts / 'evaluator_only' / trial.settings['task_id'] / 'baseline/result.json'")
    s=replace_exact(s, "['Task', 'httpx_3672', 'One previously qualified development issue.']",
                    "['Task', trial.settings.get('task_id'), 'Selected cross-task development issue.']")
    s=replace_exact(s, "    holds = [h for h in trial.holds if h['cell'] <= cell]",
                    "    def stage_order(value):\n        return 5.5 if str(value) == '5B' else float(value)\n    holds = [h for h in trial.holds if stage_order(h['cell']) <= stage_order(cell)]".replace('\n','\n'))
    sources['review.py']=s
    for name,content in sources.items():
        compile(content,name,'exec')
    return sources

SOURCE_GP02B = make_gp02b_sources(SOURCE_V24)


# v2.7: roll back forced investigation tools; preserve protected-test path checks
# and researcher tool audit, including the original callable ADK signatures.
def remove_forced_investigation_gate(source_map):
    sources = dict(source_map)
    s = sources['tool_gate.py']
    s = replace_exact(s,
        '"""GP02A v2.1: tool-level investigation gate. Gemma always selects tool arguments."""',
        '"""GP02B v2.7: observational tool audit + protected test paths, no forced investigation tools."""')
    s = replace_exact(s,
        '"""Observe actual tool execution, reject edits until required successful evidence."""',
        '"""Log tool execution and protect tests without requiring any specific tool calls."""')
    s = replace_exact(s,
        "return {'gate_passed': self.ready, 'read_source_files': sorted(self.sources),",
        "return {'gate_passed': None, 'gate_enforced': False, 'investigation_evidence_observed': self.ready, 'read_source_files': sorted(self.sources),")
    s = replace_exact(s,
        "'Call read_file and get_code_neighbors successfully before editing.'}",
        "'Choose a non-protected path; repository tests must not be modified.'}")
    s = replace_exact(s,
        "                missing = ([part for part, passed in\n"
        "                            [('successful read_file', bool(self.sources)),\n"
        "                             ('successful get_code_neighbors with nonempty neighbors', bool(self.graph_nodes))]\n"
        "                            if not passed])\n", "")
    s = replace_exact(s,
        "                    elif not self.ready:\n"
        "                        denied = self._blocked('Mandatory investigation is incomplete: source modifications are blocked.', missing)\n", "")
    s = replace_exact(s,
        "                    if not self.ready and not safe_discovery_command(command):\n"
        "                        denied = self._blocked('Before investigation completes, run_command is restricted to read-only discovery.', missing)\n", "")
    s = replace_exact(s,
        "                    # Detect obvious test-file mutation attempts after gate; not a complete OS sandbox.\n"
        "                    elif re.search",
        "                    # Detect obvious command-based test-file mutation attempts; not a complete OS sandbox.\n"
        "                    if re.search")
    # The earlier shell test-file protection uses double-escaped regex tokens.
    # Make the EXISTING prohibition effective without imposing investigation prerequisites.
    s = replace_exact(s,
        r"r'(?:>|>>|\\bsed\\s+-i\\b|\\btee\\b|\\btouch\\b)'",
        r"r'(?:>|>>|\bsed\s+-i\b|\btee\b|\btouch\b)'")
    s = replace_exact(s,
        r"r'(?:tests?/|test_[^\\s/]*\\.py)'",
        r"r'(?:tests?/|test_[^\s/]*\.py)'")
    s = replace_exact(s,
        "                elif _name == 'submit_patch' and not self.ready:\n"
        "                    denied = self._blocked('Mandatory investigation is incomplete: patch submission is blocked.', missing)\n", "")
    assert 'Mandatory investigation is incomplete' not in s
    assert 'elif not self.ready' not in s
    assert 'PROTECTED_PATH.search(path)' in s and 'ProtectedPathDenied' in s
    sources['tool_gate.py'] = s

    s = sources['smoke_worker.py']
    s = replace_exact(s,
        '        # GP02A v2.1: install a deterministic gate at the actual tool registry boundary.\n'
        '        # All arguments still originate with Gemma; the controller does NOT pre-fetch graph evidence.',
        '        # v2.7: audit actual tool execution and protect tests; no mandatory\n'
        '        # read_file or graph calls. The controller does NOT pre-fetch graph evidence.')
    s = replace_exact(s, "'mandatory_investigation_gate_passed': investigation_gate.ready,",
                      "'investigation_evidence_observed': investigation_gate.ready,\n"
                      "                      'investigation_gate_enforced': False,")
    s = replace_exact(s,"'mandatory_graph_nodes': sorted(investigation_gate.graph_nodes),",
                      "'observed_graph_nodes': sorted(investigation_gate.graph_nodes),")
    s = replace_exact(s,"'mandatory_graph_blocked_calls': dict(investigation_gate.blocked)",
                      "'policy_blocked_tool_calls': dict(investigation_gate.blocked)")
    s = replace_exact(s,"'mandatory_gate': investigation_gate.snapshot(),",
                      "'tool_policy_audit': investigation_gate.snapshot(),")
    sources['smoke_worker.py'] = s

    s = sources['workflow.py']
    s = replace_exact(s, "'mandatory_code_graph': True,", "'mandatory_code_graph': False,")
    s = replace_exact(s, "'mandatory_investigation_gate_passed': agent.get('mandatory_investigation_gate_passed', False),",
                      "'investigation_evidence_observed': agent.get('investigation_evidence_observed', False),\n"
                      "            'investigation_gate_enforced': False,")
    s = replace_exact(s, "'mandatory_graph_nodes': agent.get('mandatory_graph_nodes', []),",
                      "'observed_graph_nodes': agent.get('observed_graph_nodes', []),")
    s = replace_exact(s, "'mandatory_gate_blocked_calls': agent.get('mandatory_graph_blocked_calls', {}),",
                      "'policy_blocked_tool_calls': agent.get('policy_blocked_tool_calls', {}),")
    s = replace_exact(s, "                                            and agent.get('mandatory_investigation_gate_passed', False)\n", "")
    sources['workflow.py'] = s
    for name, code in sources.items():
        compile(code, name, 'exec')
    return sources

SOURCE_GP02B = remove_forced_investigation_gate(SOURCE_GP02B)
assert SOURCE_GP02B['tool_gate.py'] != SOURCE_V24['tool_gate.py'], 'Mandatory tool gate was not rolled back.'
assert SOURCE_GP02B['evaluator_worker.py'] == SOURCE_V24['evaluator_worker.py'], 'Protected evaluator changed.'
assert all(PROMPTS_V24[k] == PROMPTS_V24[k] for k in PROMPT_ORDER)
GP02B_SOURCE_DIR=GP02B_ROOT/'task_adapted_sources'
GP02B_SOURCE_DIR.mkdir(exist_ok=True)
for name, code in SOURCE_GP02B.items():
    (GP02B_SOURCE_DIR/name).write_text(code,encoding='utf-8')
source_manifest={name:{'sha256':hashlib.sha256(code.encode()).hexdigest(),
                       'same_as_gp02a_v24':code==SOURCE_V24[name]}
                 for name,code in SOURCE_GP02B.items()}
(GP02B_ROOT/'source_adapter_manifest.json').write_text(json.dumps(source_manifest,indent=2))
assert sum(not m['same_as_gp02a_v24'] for m in source_manifest.values())==5
print('GP02B CPU SOURCE ADAPTER: PASS — 5 harness/view files adapted; no mandatory investigation prerequisite')
print('Original v2.4 prompt SHA256 preserved:', PROMPT_SHA)
print('v2.7 prompt SHA256:', PROMPT_SHA_V27)

## GP02B 04 — Confirm current vLLM and load the adapted Trial class (no reload)

Use this notebook's serving process with the archived v2.4 configuration. Changing the selected task does not reset loaded vLLM weights or alter the model sampling policy.

In [ ]:
import requests
model_endpoint = server_instance.base_url.rstrip('/') + '/models'
resp=requests.get(model_endpoint,headers={'Authorization':'Bearer '+__import__('os').environ.get('LOCAL_API_KEY','EMPTY')},timeout=15)
resp.raise_for_status()
assert isinstance(resp.json().get('data'),list), 'vLLM returned unexpected model listing'
assert len(resp.json()['data'])>=1

workflow_path=GP02B_SOURCE_DIR/'workflow.py'
spec=importlib.util.spec_from_file_location('gp02b_frozen_v24_task_adapter', workflow_path)
workflow_gp02b=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=workflow_gp02b
spec.loader.exec_module(workflow_gp02b)
print('GP02B RUNTIME: PASS | current kernel server:',server_instance.base_url)
print('No restart during task switching.')

## GP02B 05 — Choose tasks and execute sequentially

**Default:** The registration section sets `TASKS_TO_RUN` to all six selected development tasks. Section 06 runs each sequentially with the revised tool retrieval limits. Each run uses fresh clean workspaces, a separate log and verifier. Reference qualification is mandatory but evaluator-only. Stop a task on infrastructure hold and preserve its evidence; continue with another task only when appropriate.

To run the entire cohort later, use `TASKS_TO_RUN = list(FROZEN_TASK_MANIFEST)`; previously recorded task outcomes are not overwritten.

In [ ]:
TASKS_TO_RUN = ['rich_2943', 'fastapi_5624', 'fastapi_14479']  # Initial registered tasks; Section 05C selects all six.
assert TASKS_TO_RUN and len(TASKS_TO_RUN)==len(set(TASKS_TO_RUN))
assert all(k in FROZEN_TASK_MANIFEST for k in TASKS_TO_RUN), 'Only frozen development task IDs allowed.'

# Revised tool limits use their own result register, preserving all earlier v2.4 results.
BATCH_CSV=GP02B_ROOT/'development_baseline_results_v27.csv'
BATCH_JSON=GP02B_ROOT/'development_baseline_results_v27.json'

TOOL_LIMITS_V25 = {'max_file_lines': 150, 'max_file_chars': 8000, 'max_stdout_chars': 6000}

def get_batch_results():
    return json.loads(BATCH_JSON.read_text()) if BATCH_JSON.exists() else {}

def save_batch_results(rows):
    import csv
    BATCH_JSON.write_text(json.dumps(rows,indent=2,default=str))
    keys=['task_id','status','qualified','independent_test_passed','independent_test_total',
          'elapsed_seconds','model_calls','tool_calls','patch_bytes','submitted',
          'run_id','run_dir','archive','error','trace_explorer','trace_explorer_error']
    with BATCH_CSV.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=keys,extrasaction='ignore')
        w.writeheader()
        for t in FROZEN_TASK_MANIFEST:
            if t in rows: w.writerow(rows[t])

def make_task_trial(task_id):
    expected=FROZEN_TASK_MANIFEST[task_id]
    settings=copy.deepcopy(SETTINGS_V24)
    # Retain the v2.5 tool-retrieval caps unchanged.
    settings.update(TOOL_LIMITS_V25)
    settings['expected_composed_system_sha256'] = PROMPT_SHA_V27
    settings.update(task_id=task_id,task_expected_repo=expected['repo'],
                    task_expected_commit=expected['base_commit'],
                    output_base=str(GP02B_ROOT),
                    starter_server={
                      'config':dataclasses.asdict(vllm_cfg),
                      'base_url':server_instance.base_url,
                      'command':server_instance.build_cmd(),
                      'log_path':server_instance.log_path,
                      'lifecycle_owner':'official_starter'})
    assets=dict(BASE_ASSETS)
    assets.update(sources=SOURCE_GP02B, protocols=PROMPTS_V27,
                  snapshot_sha256=SNAPSHOT_PINS[task_id])
    # Never reuse old HTTPX evaluator controls for another task.
    for stale in ('cpu_qualification_summary','cpu_evaluator_directory','cpu_environment_fingerprint'):
        assets.pop(stale,None)
    return workflow_gp02b.Trial(settings,assets)

print('Selected tasks:',TASKS_TO_RUN)
print('Frozen v2.4 reference prompt:',PROMPT_SHA)
print('Active v2.7 prompt:',PROMPT_SHA_V27)
print('Model ready; repair has NOT started in this cell.')
# Starter configuration is restored automatically at the beginning of Section 06.
# Do not require a manual assertion or a GP02A runtime object.
print('Starter inputs will be restored automatically when Section 06 starts.')


## GP02B 05A — Register GP00 reserve development task for Experiment 05

Run this CPU-only registration after Section 05 has prepared its frozen task manifest and trial factory.
It selects `fastapi_15589`, verifies repository and commit identity, checks all 36 evaluation IDs
for overlap, and records a separate amendment without overwriting earlier experiments.

**Stop on a registration error.** Stage 5 still performs original/reference qualification before allowing Gemma repair.


In [ ]:
# Experiment 05 is selected by default. For other GP02B tasks, change this to False.
USE_EXP05_RESERVE = True
if USE_EXP05_RESERVE:
    # GP02B — Register one reserved development task (CPU-only).
    # Run in the EXISTING GP02B Kaggle notebook AFTER its setup cells,
    # and BEFORE the copied Section 06 execution cell.
    # This cell does not call Gemma, restart vLLM, edit the frozen v2.4 agent,
    # or overwrite results from previous trials.

    from pathlib import Path
    import hashlib
    import json
    from datetime import datetime, timezone

    reserve_id = 'fastapi_15589'
    expected_repo = 'fastapi/fastapi'
    expected_commit = '22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b'

    # GP00 predeclared reserve, selected for its bounded, explicit bug-fix scope.
    # It is NOT one of the original 36 evaluation candidates.
    protected_eval_ids = {
        'fastapi_14964', 'fastapi_15023', 'requests_7505', 'requests_7502',
        'requests_7433', 'requests_7427', 'requests_7315', 'requests_7328',
        'requests_7309', 'requests_7205', 'rich_4070', 'rich_4077',
        'fastapi_14258', 'fastapi_14794', 'fastapi_14463', 'fastapi_12942',
        'fastapi_14482', 'fastapi_14458', 'fastapi_14266', 'rich_3718',
        'rich_3953', 'rich_3944', 'fastapi_15763', 'fastapi_15745',
        'fastapi_14303', 'requests_6644', 'requests_6589', 'requests_6592',
        'rich_3772', 'rich_3535', 'rich_3506', 'rich_3470', 'rich_3278',
        'rich_3043', 'rich_3067', 'rich_3006',
    }

    if reserve_id in protected_eval_ids or len(protected_eval_ids) != 36:
        raise RuntimeError('Development/evaluation selection must be reviewed.')

    expected_original = {'rich_2943', 'rich_3130', 'fastapi_5624', 'fastapi_14479'}
    if not expected_original.issubset(FROZEN_TASK_MANIFEST):
        raise RuntimeError('Original four GP02B registrations are missing; stop.')

    # Retrieve ONLY public task fields; never send reference patches or protected
    # test content to the repair agent.
    public_row = None
    with (COMPETITION / 'tasks.jsonl').open(encoding='utf-8') as stream:
        for line in stream:
            item = json.loads(line)
            if item.get('instance_id') == reserve_id:
                public_row = {
                    key: item[key]
                    for key in ('instance_id', 'repo', 'base_commit',
                                'problem_statement', 'created_at')
                    if key in item
                }
                break

    if not public_row:
        raise RuntimeError('Reserve not found in the current published tasks.jsonl.')
    if (public_row['repo'], public_row['base_commit']) != (expected_repo, expected_commit):
        raise RuntimeError('Reserve repo/commit differs from the GP00 reserve register.')
    if any(
        other.get('repo') == expected_repo and other.get('base_commit') == expected_commit
        for task, other in FROZEN_TASK_MANIFEST.items()
        if task != reserve_id
    ):
        raise RuntimeError('A development task already uses this repository snapshot.')

    # Check that none of the 36 evaluation tasks shares the reserve's source
    # repository and base commit. This uses public metadata only.
    evaluation_groups = set()
    with (COMPETITION / 'tasks.jsonl').open(encoding='utf-8') as stream:
        for line in stream:
            item = json.loads(line)
            if item.get('instance_id') in protected_eval_ids:
                evaluation_groups.add((item['repo'], item['base_commit']))
    if (expected_repo, expected_commit) in evaluation_groups:
        raise RuntimeError('Reserve overlaps the original evaluation split.')

    snapshot = COMPETITION / 'snapshots' / (reserve_id + '.tgz')
    if not snapshot.is_file():
        raise RuntimeError('Reserve snapshot missing: ' + str(snapshot))
    snapshot_hash = hashlib.sha256(snapshot.read_bytes()).hexdigest()

    # Add a new study-version amendment. Do not overwrite the original four-task
    # frozen_development_cohort.json or the four recorded task outcomes.
    amendment = {
        'version': 'GP02B_development_reserve_amendment_v1',
        'purpose': 'Replace requests_6757 infrastructure hold for development only',
        'previously_frozen_development_tasks': sorted(expected_original),
        'original_holds': ['rich_3130', 'requests_6757'],
        'reserve_task_id': reserve_id,
        'reserve_repo': expected_repo,
        'reserve_base_commit': expected_commit,
        'reserve_snapshot_sha256': snapshot_hash,
        'selection_source': 'GP00/02_candidate_split.csv: original reserve row',
        'selection_reason': 'Bounded query-header behaviour, selected from public issue text before testing',
        'original_36_evaluation_candidate_ids': sorted(protected_eval_ids),
        'system_prompt_sha256': PROMPT_SHA,
        'v24_agent_changed': False,
    }

    amendment_file = GP02B_ROOT / 'development_reserve_amendment_fastapi_15589.json'
    if amendment_file.is_file():
        existing = json.loads(amendment_file.read_text(encoding='utf-8'))
        if existing != amendment:
            raise RuntimeError('Existing reserve amendment differs; do not overwrite.')
    else:
        amendment_file.write_text(json.dumps(amendment, indent=2) + '\n', encoding='utf-8')

    FROZEN_TASK_MANIFEST[reserve_id] = {
        'repo': expected_repo,
        'base_commit': expected_commit,
    }
    TASK_ROWS[reserve_id] = dict(public_row)
    PUBLIC_TASKS[reserve_id] = dict(public_row)
    SNAPSHOT_PINS[reserve_id] = snapshot_hash
    TASKS_TO_RUN = [reserve_id]

    print('DEVELOPMENT RESERVE: REGISTERED')
    print('Task:', reserve_id)
    print('Repository:', expected_repo)
    print('Snapshot SHA256:', snapshot_hash)
    print('Evaluation overlap: NONE (36 original evaluation IDs checked)')
    print('Frozen v2.4 prompt:', PROMPT_SHA)
    print('Previous GP02B results: UNCHANGED')
    print('vLLM server: UNCHANGED')
    print('Run the existing copied Section 06 cell next.')
    print('Stage 5 must qualify before Gemma starts a repair.')
else:
    print("Experiment 05 reserve disabled; preserving TASKS_TO_RUN from Section 05.")


## GP02B 05B — Install researcher-only HTML trace explorer

The following file is a standard-library-only renderer. It processes recorded evidence **after**
execution and produces a standalone, searchable HTML file for each run. It does not call the
model, change any agent tools, modify the prompt, consume the agent budget, or access reference
patches. The HTML includes recorded reasoning, tool arguments/results, timing, initial prompts,
and explicit limitations about unavailable exact model request payloads.

**Automatic packaging:** Section 06 writes the report into `artifacts/reports/` **before** `trial_b.export()`,
so the existing export includes the HTML, renderer source, and integrity metadata.


In [ ]:
%%writefile /kaggle/working/GP02B_trace_explorer.py
"""Researcher-only, offline GP02B Gemma trace explorer.

Reads post-run evidence from the frozen v2.4 harness. Never imports an agent,
invokes a model, calls protected tests, or mutates trial execution evidence.
Standard-library only. The report is a derived interpretation, not a serialized
per-turn model-request capture.
"""
from __future__ import annotations

import hashlib
import html
import json
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

CSS = '\n:root {color-scheme:light;--txt:#192637;--muted:#62748c;--line:#d9e1ec;--bg:#f6f8fc;--accent:#0a60b3;--surface:white}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--txt);font:14px/1.55 system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif}.layout{max-width:1350px;margin:0 auto;padding:28px 22px 80px}h1{margin:0 0 6px;font-size:29px;letter-spacing:-.7px}h2{margin:28px 0 12px;font-size:19px}h3{margin:0 0 10px;font-size:16px}h4{margin:0 0 8px;font-size:13px;color:#28384c}p{max-width:1100px}.muted{color:var(--muted)}.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin:22px 0}.kpi{border:1px solid var(--line);background:var(--surface);padding:16px;border-radius:10px}.kpi b{font-size:24px;display:block}.kpi span{font-size:12px;color:var(--muted)}.note{border:1px solid #b9d2e8;background:#eef6ff;padding:14px 17px;border-radius:10px}.warn{border:1px solid #f0d4ab;background:#fff8eb;padding:12px;border-radius:8px}.control{position:sticky;top:0;z-index:5;background:#f6f8fce8;backdrop-filter:blur(8px);padding:12px 0;border-bottom:1px solid var(--line);display:flex;gap:12px;align-items:center;flex-wrap:wrap}.control input,.control select{border:1px solid #c4d1e3;border-radius:7px;padding:9px;background:#fff;min-width:210px;font-size:14px}.control span{color:var(--muted)}.turn{border:1px solid var(--line);border-radius:12px;padding:17px 19px;margin:17px 0;background:var(--surface);box-shadow:0 2px 6px #1233}.turnnum{color:var(--accent)}.time{font-weight:400;font-size:13px;color:var(--muted);margin-left:10px}.summarytools{font-size:12px;padding-left:8px;color:#1d5a76}.metrics{display:flex;gap:17px;flex-wrap:wrap;font-size:12px;color:var(--muted);margin-bottom:14px}.metrics b{color:var(--txt)}.twocol{display:grid;grid-template-columns:1fr 1fr;gap:16px}details{border:1px solid var(--line);border-radius:7px;overflow:hidden;margin:7px 0;background:#fff}summary{background:#f4f7fb;cursor:pointer;padding:8px 11px;color:#273e55;font-weight:600}pre{white-space:pre-wrap;overflow-wrap:anywhere;max-height:580px;overflow:auto;font:12px/1.45 ui-monospace,SFMono-Regular,Consolas,monospace;padding:12px;margin:0;background:#fff;color:#192637}.tool{border-left:3px solid #c6d9ed;padding:10px 12px;margin-bottom:12px;background:#fafcff}.toolhead{display:flex;justify-content:space-between;gap:10px}.tiny{font-size:12px;color:var(--muted)}.chip{border-radius:999px;background:#e7eff8;color:#28567b;font-size:11px;font-weight:600;padding:3px 8px;margin-left:5px}.errchip{background:#fff0e2;color:#935019}.footer{text-align:center;color:var(--muted);font-size:12px;margin-top:30px}@media(max-width:850px){.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.twocol{grid-template-columns:1fr}.time,.summarytools{display:block;margin-left:0;padding-left:0}}\n'


def _hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None


def _read_json(path, default=None):
    try:
        return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else default
    except (OSError, ValueError, UnicodeError):
        return default


def _read_text(path):
    try:
        return path.read_text(encoding='utf-8') if path.is_file() else ''
    except (OSError, UnicodeError):
        return ''


def _json_lines(path):
    out = []
    if path.is_file():
        for number, raw in enumerate(_read_text(path).splitlines(), 1):
            if raw.strip():
                try:
                    row = json.loads(raw)
                    if isinstance(row, dict):
                        out.append(row)
                except ValueError:
                    out.append({'type': 'trace_parse_warning', 'detail': f'Invalid JSON at line {number}'})
    return out


def _detail(value):
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except ValueError:
        return value


def _pretty(value):
    if value is None:
        return '(not recorded)'
    if isinstance(value, str):
        return value
    try:
        return json.dumps(value, ensure_ascii=False, indent=2, default=str)
    except (ValueError, TypeError):
        return str(value)


def _e(value):
    return html.escape(str(value), quote=True)


def _safe_slug(value):
    return re.sub(r'[^a-zA-Z0-9_-]+', '_', str(value)).strip('_')[:100] or 'unknown'


def _seconds(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _display_seconds(value):
    return f'{value:.1f}s' if isinstance(value, (int, float)) else 'unavailable'


def _is_error(result):
    if isinstance(result, dict):
        state = str(result.get('status', '')).lower()
        return bool((state and state not in ('ok', 'success', 'passed'))
                    or result.get('error') or result.get('exception')
                    or result.get('is_error') is True)
    return False


def _new_turn(event, seq):
    return {'number': seq, 'event_id': event.get('event_id'),
            't': _seconds(event.get('elapsed_seconds')),
            'record': _detail(event.get('detail')),
            'usage': None, 'thoughts': [], 'assistant_text': [],
            'tools': [], 'extras': [], 'last_prior_results': []}


def _build_turns(events, model_responses, legacy):
    turns = []
    pending = []
    stray = []
    for event in events:
        typ = str(event.get('type', ''))
        val = _detail(event.get('detail'))
        if typ == 'model_turn':
            turns.append(_new_turn(event, len(turns) + 1))
            continue
        if typ == 'tool_call':
            if not turns:
                turns.append(_new_turn({}, 1))
            call = {'name': event.get('tool') or '(unnamed tool)',
                    'call_id': event.get('call_id'),
                    'event_id': event.get('event_id'),
                    'time': _seconds(event.get('elapsed_seconds')),
                    'args': val, 'response': None, 'response_time': None}
            turns[-1]['tools'].append(call)
            pending.append(call)
            continue
        if typ == 'tool_result':
            call_id = event.get('call_id')
            tool = event.get('tool')
            match = next((c for c in pending if call_id and c['call_id'] == call_id), None)
            if match is None:
                match = next((c for c in pending if tool and c['name'] == tool), None)
            if match is None and len(pending) == 1:
                match = pending[0]
            if match is not None:
                match['response'] = val
                match['response_time'] = _seconds(event.get('elapsed_seconds'))
                pending.remove(match)
            else:
                stray.append({'type': 'unpaired_tool_result', 'tool': tool, 'detail': val})
            continue
        if not turns:
            if typ in ('agent_error', 'worker_error', 'trace_parse_warning'):
                stray.append({'type': typ, 'detail': val})
            continue
        if typ == 'model_thought':
            turns[-1]['thoughts'].append(_pretty(val))
        elif typ == 'assistant_text':
            turns[-1]['assistant_text'].append(_pretty(val))
        elif typ == 'model_usage':
            turns[-1]['usage'] = val
        elif typ in ('agent_error', 'worker_error', 'context_warning', 'trace_parse_warning'):
            turns[-1]['extras'].append({'type': typ, 'detail': val})

    legacy_thoughts = [entry for entry in (legacy.get('entries') or [])
                       if isinstance(entry, dict) and entry.get('type') == 'thinking']
    records = model_responses if isinstance(model_responses, list) else []
    if not turns:
        # A partial/held run may lack ADK events. Show only actual recorded
        # response entries. Never invent tool calls or timings.
        for record in records or legacy_thoughts:
            turns.append(_new_turn({'event_id': None,
                         'elapsed_seconds': record.get('elapsed_seconds', record.get('elapsed_s'))},
                         len(turns) + 1))

    if len(records) == len(turns):
        for turn, record in zip(turns, records):
            txt = record.get('exposed_reasoning')
            if txt and not turn['thoughts']:
                turn['thoughts'] = [_pretty(txt)]
            if not turn['usage']:
                turn['usage'] = record.get('usage')
            if turn['t'] is None:
                turn['t'] = _seconds(record.get('elapsed_seconds'))
    elif len(legacy_thoughts) == len(turns):
        for turn, entry in zip(turns, legacy_thoughts):
            if not turn['thoughts'] and entry.get('content'):
                turn['thoughts'] = [_pretty(entry['content'])]
            if not turn['usage']:
                turn['usage'] = entry.get('usage')
            if turn['t'] is None:
                turn['t'] = _seconds(entry.get('elapsed_s'))

    for index, turn in enumerate(turns):
        if index:
            previous = turns[index - 1]
            turn['last_prior_results'] = [(x['name'], x['response']) for x in previous['tools']
                                          if x['response'] is not None]
    return turns, stray, len(pending)


def _usage_tokens(usage):
    if not isinstance(usage, dict):
        return None, None
    inp = usage.get('prompt_token_count', usage.get('prompt_tokens'))
    out = usage.get('candidates_token_count', usage.get('completion_tokens'))
    return inp, out


def _details(label, body, opened=False):
    return f'<details{" open" if opened else ""}><summary>{_e(label)}</summary><pre>{_e(body)}</pre></details>'


def _html_turn(turn, previous):
    num = turn['number']
    gap = turn['t'] - previous['t'] if previous and turn['t'] is not None and previous['t'] is not None else None
    types = [c['name'] for c in turn['tools']]
    flags = []
    if any(x == 'edit_file' for x in types): flags.append('Edit')
    if any(x == 'write_file' for x in types): flags.append('Write')
    if any(x == 'run_command' for x in types): flags.append('Command')
    if any(x.startswith('get_code_') for x in types): flags.append('Graph')
    if any(x == 'search_similar_code' for x in types): flags.append('RAG')
    if any(x == 'submit_patch' for x in types): flags += ['Submit', 'Final']
    if any(_is_error(c['response']) for c in turn['tools']): flags.append('Error')
    if turn['extras'] and any(e['type'] in ('agent_error', 'worker_error') for e in turn['extras']): flags.append('Error')
    chips = ''.join(f'<span class="chip{" errchip" if f == "Error" else ""}">{_e(f)}</span>' for f in flags)
    inp, out = _usage_tokens(turn['usage'])
    headline = (f'<h3><span class="turnnum">Turn {num:02d}</span>'
                f'<span class="time">t={_display_seconds(turn["t"])}</span>'
                f'<span class="time">~{_display_seconds(gap)} since prior response</span>'
                f'<span class="summarytools">{_e(", ".join(types) or "No tool request")}</span>{chips}</h3>')
    metrics = (f'<div class="metrics"><span>Model output observed at: <b>{_display_seconds(turn["t"])}</b></span>'
               f'<span>Approximate response gap: <b>{_display_seconds(gap)}</b></span>'
               f'<span>Prompt tokens: <b>{_e(inp if inp is not None else "not recorded")}</b></span>'
               f'<span>Completion tokens: <b>{_e(out if out is not None else "not recorded")}</b></span></div>')
    if turn['last_prior_results']:
        preceding = '\n\n'.join(f'TOOL: {name}\n{_pretty(result)}' for name, result in turn['last_prior_results'])
        before = _details(f'Immediately preceding tool responses ({len(turn["last_prior_results"])}) — not exact serialized model input', preceding)
    elif num == 1:
        before = '<p class="muted">First recorded turn. Initial system/task prompts, when available, appear above.</p>'
    else:
        before = '<p class="muted">No preceding tool response was recorded for the previous turn.</p>'
    reasoning = '\n\n'.join(turn['thoughts']).strip()
    assistant = '\n\n'.join(turn['assistant_text']).strip()
    thought_html = (_details('View entire recorded reasoning', reasoning, opened=(num == 1)) if reasoning
                    else '<p class="muted">No exposed reasoning was recorded for this response.</p>')
    if assistant:
        thought_html += _details('Assistant text', assistant)
    tools_html = ''
    for call in turn['tools']:
        response_t = call['response_time']
        delta = response_t - call['time'] if response_t is not None and call['time'] is not None else None
        tools_html += (f'<div class="tool"><div class="toolhead"><b>{_e(call["name"])}</b>'
                       f'<span class="tiny">call at {_display_seconds(call["time"])}</span></div>'
                       + _details('Tool request arguments', _pretty(call['args']), opened=(num == 1))
                       + f'<div class="tiny">Response {_display_seconds(response_t)} · observed event delta {_display_seconds(delta)}'                       + '</div>'
                       + _details('Recorded tool result (expand)', _pretty(call['response'])) + '</div>')
    if not tools_html: tools_html = '<p class="muted">No tool request recorded in this response.</p>'
    extras = ''.join(_details(f'Recorded {x["type"]}', _pretty(x['detail'])) for x in turn['extras'])
    return (f'<section class="turn" id="turn-{num}" data-flags="{_e(" ".join(flags))}">{headline}{metrics}'
            f'<div class="twocol"><div><h4>Before this response: recorded context</h4>{before}</div>'
            f'<div><h4>Gemma’s recorded reasoning / thinking</h4>{thought_html}</div></div>'
            f'<div><h4>Tool requests and returned results</h4>{tools_html}</div>{extras}</section>')


def build_trace_explorer(artifacts, task_id=None, run_id=None, status=None,
                         output=None, protocol_dir=None):
    """Generate a standalone HTML report and derived-evidence manifest.

    Writes *only* reports/ (and the supplied HTML output). Invoke AFTER all
    experiment/evaluator stages and BEFORE trial.export() so export includes it.
    """
    artifacts = Path(artifacts)
    agent = artifacts / 'agent'
    summary = _read_json(artifacts / '00_summary.json', {}) or {}
    result = _read_json(agent / 'result.json', {}) or {}
    task_id = task_id or summary.get('task_id') or 'unknown_task'
    run_id = run_id or summary.get('run_id') or 'unknown_run'
    status = status or summary.get('status') or result.get('stage') or 'NOT_RECORDED'
    full = agent / 'events_full.jsonl'
    compact = agent / 'events.jsonl'
    event_path = full if full.is_file() else compact
    events = _json_lines(event_path)
    model_responses = _read_json(agent / 'model_responses.json', []) or []
    legacy = _read_json(agent / 'trace.json', {}) or {}
    turns, stray, unresolved = _build_turns(events, model_responses, legacy)
    tool_count = sum(len(t['tools']) for t in turns)
    response_count = sum(c['response'] is not None for t in turns for c in t['tools'])
    maximum = max((e.get('elapsed_seconds') for e in events if isinstance(e.get('elapsed_seconds'), (int,float))), default=None)
    junit = summary.get('candidate_junit') or {}
    if not isinstance(junit, dict): junit = {}
    pass_count = junit.get('passed')
    total = junit.get('tests')
    results_label = f'{pass_count}/{total}' if pass_count is not None and total is not None else 'N/A'
    proto_dir = Path(protocol_dir) if protocol_dir else None
    protocols = []
    proto_order = ('system_custom_base.md', 'investigation_protocol.md', 'tools_code_graph.md',
                   'tools_code_retrieval.md', 'tools_file_operations.md', 'tools_testing.md', 'tools_execution.md')
    if proto_dir and proto_dir.is_dir():
        for filename in proto_order:
            p = proto_dir / filename
            if p.is_file(): protocols.append(f'# COMPONENT: {filename}\n{_read_text(p)}')
    system_prompt = '\n\n'.join(protocols)
    task_prompt = _read_text(agent / 'initial_task_prompt.txt')
    warning = ('This is a read-only rendering of the recorded evidence, not a verbatim per-turn API request. '
               'The preceding-context pane shows only immediately preceding recorded tool responses. '
               'Inter-turn gaps include inference and orchestration, not proven model-only latency. '
               'Some tool outputs were shortened upstream by the harness. Recorded reasoning is not '
               'a guarantee of access to every internal computation. Protected evaluator data is not exposed.')
    html_turns = '\n'.join(_html_turn(turn, turns[i-1] if i else None) for i, turn in enumerate(turns))
    if not html_turns:
        html_turns = ('<div class="warn"><b>No Gemma model turns recorded.</b> '
                      'This can be legitimate for a qualification/tool-preflight hold. '
                      'Inspect the original ZIP for the reason; no turns are invented.</div>')
    warnings = ''
    if not full.is_file() and compact.is_file():
        warnings += '<div class="warn">Full events file unavailable: the compact log may truncate tool details.</div>'
    if unresolved:
        warnings += f'<div class="warn">{unresolved} tool request(s) lacked matched result records.</div>'
    if stray:
        warnings += _details('Unpaired / early events', _pretty(stray))
    status_note = f'{_e(status)} · task {_e(task_id)} · run {_e(run_id)}'
    sections = (f'<!doctype html><html lang="en"><head><meta charset="utf-8">'
                f'<meta name="viewport" content="width=device-width, initial-scale=1">'
                f'<title>GP02B · Gemma 4 repair trace explorer · {_e(task_id)}</title>'
                f'<style>{CSS}</style></head><body><main class="layout">'
                f'<h1>GP02B · Gemma 4 repair trace explorer</h1>'
                f'<p class="muted">Observed run {_e(run_id)} · task <code>{_e(task_id)}</code> · '
                f'status {_e(status)} · source: {_e(event_path.relative_to(artifacts)) if event_path.is_file() else "no event file"}</p>'
                f'<div class="kpis"><div class="kpi"><b>{len(turns)}</b><span>Captured Gemma turns</span></div>'
                f'<div class="kpi"><b>{tool_count}</b><span>Tool requests · {response_count} responses</span></div>'
                f'<div class="kpi"><b>{_display_seconds(maximum)}</b><span>Last observed event</span></div>'
                f'<div class="kpi"><b>{_e(results_label)}</b><span>Independent tests passed</span></div></div>'
                f'<div class="note"><b>Evidence boundary:</b> {_e(warning)}</div>{warnings}'
                f'<h2>Initial inputs supplied to the agent</h2>'
                + (_details('Full recorded system instruction', system_prompt) if system_prompt else
                   '<p class="muted">System instruction text was not available in this run’s reporting inputs.</p>')
                + (_details('Full recorded task prompt, budgets and environment rules', task_prompt) if task_prompt else
                   '<p class="muted">Initial task prompt not captured (possible pre-agent hold).</p>')
                + '<h2>Explore every model turn</h2>'
                + '<p>Filter by a tool, symbol, error, source edit or text; expand recorded tool results to inspect what happened.</p>'
                + '<div class="control"><input id="search" aria-label="Search turns" '
                  'placeholder="Filter: keep_alive, edit_file, Error…" oninput="apply()">'
                  '<select id="kind" onchange="apply()"><option value="">All turns</option>'
                  '<option value="Edit">Edits</option><option value="Write">File writes</option>'
                  '<option value="Command">Commands</option><option value="Graph">Code graph</option>'
                  '<option value="RAG">Code retrieval</option><option value="Error">Tool errors</option>'
                  '<option value="Final">Final/submit</option></select><span id="shown"></span>'
                  '<a href="#turn-1">First turn</a></div>'
                + html_turns
                + '<div class="footer">Researcher-only derived report from saved execution evidence; '
                  'no external libraries, servers or network requests.</div></main>'
                + '<script>function apply(){const q=document.getElementById("search").value.toLowerCase(),'
                  'k=document.getElementById("kind").value;let n=0;for(const e of document.querySelectorAll(".turn"))'
                  '{const ok=(!q||e.textContent.toLowerCase().includes(q))&&(!k||e.dataset.flags.split(" ").includes(k));'
                  'e.style.display=ok?"block":"none";if(ok)n++}'
                  'document.getElementById("shown").textContent=n+" / "+document.querySelectorAll(".turn").length+" turns";}'
                  'apply();</script></body></html>')
    report_dir = artifacts / 'reports'
    report_dir.mkdir(parents=True, exist_ok=True)
    output = Path(output) if output else report_dir / f'GP02B_Gemma_Trace_Explorer_{_safe_slug(task_id)}_{_safe_slug(run_id)}.html'
    if output.exists():
        raise FileExistsError(f'Cannot overwrite a previously generated trace explorer: {output}')
    output.write_text(sections, encoding='utf-8')
    inputs = {str(p.relative_to(artifacts)): _hash(p) for p in
              (full, compact, agent/'trace.json', agent/'model_responses.json',
               agent/'initial_task_prompt.txt', artifacts/'00_summary.json') if p.is_file()}
    manifest = {'report_type': 'GP02B research-only trace explorer', 'task_id': task_id,
                'run_id': run_id, 'status': status, 'generated_utc': datetime.now(timezone.utc).isoformat(),
                'output': str(output.relative_to(artifacts)) if output.is_relative_to(artifacts) else str(output),
                'report_sha256': _hash(output), 'source_sha256': inputs,
                'model_turns': len(turns), 'tool_requests': tool_count, 'tool_responses': response_count,
                'unmatched_tool_requests': unresolved,
                'source_note': 'Derived from post-run evidence. Not exact serialized model input.',
                'changes_to_agent_or_experiment': False}
    manifest_path = report_dir / 'trace_explorer_manifest.json'
    if manifest_path.exists():
        raise FileExistsError(f'Cannot overwrite an existing trace manifest: {manifest_path}')
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False)+'\n', encoding='utf-8')
    return output


## GP02B 05C — Register the remaining two selected tasks and freeze the six-task v2.7 run list (CPU only)

The original Section 05A registers `fastapi_15589`. This cell additionally registers the
previously qualified HTTPX development task `httpx_3672` and the selected Rich replacement
`rich_4079` (replacing the holds `rich_3130` and `rich_3454`), without changing the original
four-task frozen cohort or any earlier trial outcome. It checks the published repository,
base commit, snapshot, and disjointness from the **original 36-task evaluation set**.

**Execution order:** `httpx_3672`, `rich_2943`, `fastapi_5624`, `fastapi_14479`,
`fastapi_15589`, `rich_4079`. Every run uses the same three revised tool caps and the
already installed automatic trace explorer. No Gemma calls occur in this registration cell.


In [ ]:
# GP02B v2.7 — CPU-only six-task registration and execution selection.
# Does not modify prompts, tools, evaluator, time budgets or previous results.

from pathlib import Path
import hashlib
import json

SIX_DEV_TASK_IDS = (
    'httpx_3672',
    'rich_2943',
    'fastapi_5624',
    'fastapi_14479',
    'fastapi_15589',
    'rich_4079',
)

# Original, unchanged 36-task evaluation cohort from Section 05A.
if 'protected_eval_ids' not in globals() or len(protected_eval_ids) != 36:
    raise RuntimeError('Original 36 evaluation IDs are unavailable or altered; stop.')
if any(t in protected_eval_ids for t in SIX_DEV_TASK_IDS):
    raise RuntimeError('Development/evaluation task ID overlap; stop.')

with (COMPETITION / 'tasks.jsonl').open(encoding='utf-8') as _f:
    _rows_list = [json.loads(line) for line in _f if line.strip()]
_public_by_id = {r['instance_id']: r for r in _rows_list}
if len(_public_by_id) != len(_rows_list):
    raise RuntimeError('Duplicate task IDs in published metadata; stop.')
if not set(protected_eval_ids).issubset(_public_by_id):
    raise RuntimeError('Missing evaluation task IDs in current published inventory; stop.')
if not set(SIX_DEV_TASK_IDS).issubset(_public_by_id):
    raise RuntimeError('A selected development task is absent from tasks.jsonl; stop.')

# Add only the two tasks not already registered in the original v1.5 notebook.
_extra_repos = {'httpx_3672': 'encode/httpx', 'rich_4079': 'Textualize/rich'}
for _task_id, _expected_repo in _extra_repos.items():
    _row = _public_by_id[_task_id]
    _repo, _commit = _row.get('repo'), _row.get('base_commit')
    if _repo != _expected_repo or not _commit:
        raise RuntimeError(f'Unexpected repo/base commit for {_task_id}; stop.')
    if _task_id == 'httpx_3672' and _commit != '4acf5c2c37714cc63b5cf71b3e284fca83c90311':
        raise RuntimeError('HTTPX base commit differs from original frozen baseline; stop.')
    if _task_id in FROZEN_TASK_MANIFEST:
        if FROZEN_TASK_MANIFEST[_task_id] != {'repo': _repo, 'base_commit': _commit}:
            raise RuntimeError(f'Previously registered task identity differs: {_task_id}')
    else:
        _snapshot = COMPETITION / 'snapshots' / (_task_id + '.tgz')
        if not _snapshot.is_file():
            raise RuntimeError(f'Missing source snapshot: {_snapshot}')
        _sha = hashlib.sha256(_snapshot.read_bytes()).hexdigest()
        _public_fields = ('instance_id','repo','base_commit','problem_statement','created_at')
        _public_row = {k:_row[k] for k in _public_fields if k in _row}
        FROZEN_TASK_MANIFEST[_task_id] = {'repo': _repo, 'base_commit': _commit}
        TASK_ROWS[_task_id] = dict(_public_row)
        PUBLIC_TASKS[_task_id] = dict(_public_row)
        SNAPSHOT_PINS[_task_id] = _sha

# Check all six snapshot groups against the evaluation split and each other.
_eval_groups = {(_public_by_id[t]['repo'], _public_by_id[t]['base_commit'])
                for t in protected_eval_ids}
_selected_groups = [(FROZEN_TASK_MANIFEST[t]['repo'], FROZEN_TASK_MANIFEST[t]['base_commit'])
                    for t in SIX_DEV_TASK_IDS]
if len(set(_selected_groups)) != len(SIX_DEV_TASK_IDS):
    raise RuntimeError('Selected development tasks share a repository/base-commit group; stop.')
if set(_selected_groups) & _eval_groups:
    raise RuntimeError('A selected development snapshot overlaps the frozen evaluation cohort; stop.')
for _task_id in SIX_DEV_TASK_IDS:
    _row = _public_by_id[_task_id]
    _expect = FROZEN_TASK_MANIFEST[_task_id]
    if (_row['repo'], _row['base_commit']) != (_expect['repo'], _expect['base_commit']):
        raise RuntimeError(f'Published identity mismatch: {_task_id}')
    _snapshot = COMPETITION / 'snapshots' / (_task_id + '.tgz')
    if not _snapshot.is_file():
        raise RuntimeError(f'Snapshot missing: {_snapshot}')
    _actual_sha = hashlib.sha256(_snapshot.read_bytes()).hexdigest()
    if SNAPSHOT_PINS[_task_id] != _actual_sha:
        raise RuntimeError(f'Snapshot hash mismatch: {_task_id}')

# Record the specific six selected candidates and the *only* three revised limits.
_v25_manifest = {
    'revision': 'GP02B_v2.7_autonomous_tool_selection',
    'selected_task_ids': list(SIX_DEV_TASK_IDS),
    'previous_holds': ['rich_3130', 'requests_6757', 'rich_3454'],
    'task_snapshots': {t: {'repo': FROZEN_TASK_MANIFEST[t]['repo'],
                           'base_commit': FROZEN_TASK_MANIFEST[t]['base_commit'],
                           'sha256': SNAPSHOT_PINS[t]}
                       for t in SIX_DEV_TASK_IDS},
    'frozen_evaluation_ids': sorted(protected_eval_ids),
    'tool_limits_v24': {k: SETTINGS_V24[k] for k in TOOL_LIMITS_V25},
    'tool_limits_v25': TOOL_LIMITS_V25,
    'original_prompt_sha256': PROMPT_SHA,
    'active_prompt_sha256': PROMPT_SHA_V27,
    'changed_prompt_components': ['system_custom_base.md', 'investigation_protocol.md', 'tools_code_graph.md', 'tools_file_operations.md', 'tools_testing.md'],
    'automatic_html_trace': True,
    'selection_disclosure': ('rich_4079 selected after observing prior external qualification '
                             'and repair outcomes; this is a development selection, not evaluation tuning'),
}
_v25_path = GP02B_ROOT / 'development_six_task_manifest_v27.json'
if _v25_path.is_file():
    if json.loads(_v25_path.read_text(encoding='utf-8')) != _v25_manifest:
        raise RuntimeError('Existing v2.7 six-task manifest differs; do not overwrite.')
else:
    _v25_path.write_text(json.dumps(_v25_manifest, indent=2) + '\n', encoding='utf-8')

# This is the final authoritative six-task selection for Section 06.
TASKS_TO_RUN = list(SIX_DEV_TASK_IDS)
print('GP02B v2.7 SIX DEVELOPMENT TASKS:', TASKS_TO_RUN)
print('Tool caps:', TOOL_LIMITS_V25)
print('36-task held-out evaluation split: UNCHANGED')
print('Original v2.4 trial results: PRESERVED in separate register')
print('HTML trace: automatically generated for each run by Section 06')
print('Next: execute Section 06 once to run all six tasks sequentially.')


## GP02B 06 — Run the selected tasks (GPU + independent evaluator)

Restores the correct original v2.4 starter inputs automatically, then runs one task at a time with explicit stage checkpoints: discover → install reuse → prepare → baseline/reference qualification → ADK graph readiness → autonomous repair → independent verification → export. A failed or held task is recorded rather than mislabelled as a repaired issue. Monitor the existing per-model-call and per-tool-call live trace inside each run.

In [ ]:
import time
# Re-create the v2.4 starter inputs on each new batch run.
# This is required because the archive also includes already-processed output files.
from zipfile import ZipFile
_original_starter_files = {
    'google_original_agent.yaml': 'agent.yaml',
    'google_original_eval_config.yaml': 'eval_config.yaml',
    'prompts/google_original_system.md': 'prompts/system.md',
    'configs/google_original_sampling.yaml': 'configs/sampling.yaml',
}
with ZipFile(BASELINE_ZIP) as _archive:
    for _source, _dest in _original_starter_files.items():
        _target = AGENT_DIR / _dest
        _target.parent.mkdir(parents=True, exist_ok=True)
        _target.write_bytes(_archive.read('run_sources/submission/' + _source))
print('Restored original v2.4 starter inputs. Previous evidence files remain untouched.')

batch_rows=get_batch_results()
for task_id in TASKS_TO_RUN:
    if task_id in batch_rows:
        prior = batch_rows[task_id]
        if prior.get('status') == 'PATCH_VERIFIED':
            print(f'[SKIP] {task_id}: verified checkpoint {prior["status"]}.')
            continue
        # Previous attempt remains archived at prior['archive']; clear only the
        # register entry to allow an explicit user-initiated rerun of Cell 19.
        print(f'[RETRY] {task_id}: previous {prior["status"]} '
              f'(archive preserved at {prior.get("archive")}).')
        batch_rows.pop(task_id)
        save_batch_results(batch_rows)
    print('\n'+'='*76)
    print('GP02B TASK:',task_id,'|',FROZEN_TASK_MANIFEST[task_id]['repo'])
    print('Active v2.7 prompt:',PROMPT_SHA_V27,'| max_output_tokens=2048 | 15-min agent budget')
    print('='*76,flush=True)
    started=time.monotonic()
    trial_b=make_task_trial(task_id)
    step_labels=[(2,trial_b.discover),(3,trial_b.install),(4,trial_b.prepare),
                 (5,trial_b.qualify),('5B',trial_b.tool_preflight),
                 (6,trial_b.repair),(7,trial_b.verify)]
    for stage,action in step_labels:
        print(f'\n[{task_id}] START STAGE {stage}',flush=True)
        trial_b.step(stage,action)
        if trial_b.holds:
            print(f'[{task_id}] HOLD at stage {stage}: {trial_b.holds[-1]["reason"]}',flush=True)
            break
        if stage==5 and not trial_b.qualified:
            print('QUALIFICATION_HOLD — no model repair attempted.',flush=True)
            break
        if stage=='5B' and not trial_b.tools_ready:
            print('TOOL_PREFLIGHT_HOLD — no model repair attempted.',flush=True)
            break
    try:
        summary=trial_b.summarize()
    except Exception as exc:
        print('SUMMARY HOLD:',repr(exc))
        summary={'status':'REVIEW_REQUIRED','run_id':trial_b.run_id,'error':repr(exc)}
    # Task wall time is fixed BEFORE researcher report generation or ZIP
    # compression. HTML processing must never inflate experiment timings.
    experiment_elapsed_seconds = round(time.monotonic() - started, 2)
    # ------------------------------------------------------------
    # Researcher reporting ONLY. This occurs after all model/evaluator
    # stages and summarization, and before the existing ZIP export.
    # The generated HTML and renderer source are immutable ZIP inputs.
    # Reporting failure NEVER changes an agent/test verdict.
    # ------------------------------------------------------------
    trace_explorer_path = None
    trace_explorer_error = None
    try:
        import shutil
        _report_source = Path('/kaggle/working/GP02B_trace_explorer.py')
        if not _report_source.is_file():
            raise FileNotFoundError('Run the GP02B 05B trace-reporter cell before Section 06.')
        _spec = importlib.util.spec_from_file_location('gp02b_trace_explorer', _report_source)
        _report_module = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_report_module)
        _report_dir = trial_b.artifacts / 'reports'
        _report_dir.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(_report_source, _report_dir / 'trace_explorer_generator.py')
        trace_explorer_path = _report_module.build_trace_explorer(
            trial_b.artifacts, task_id=task_id,
            run_id=trial_b.run_id, status=summary.get('status'),
            protocol_dir=trial_b.root / 'protocol')
        print('[TRACE EXPLORER] Created:', trace_explorer_path, flush=True)
    except Exception as exc:
        trace_explorer_error = type(exc).__name__ + ': ' + str(exc)
        print('[TRACE EXPLORER REPORTING HOLD]', trace_explorer_error,
              '(experiment outcome unchanged)', flush=True)
        _error_file = trial_b.artifacts / 'reports' / 'trace_explorer_error.json'
        _error_file.parent.mkdir(parents=True, exist_ok=True)
        _error_file.write_text(json.dumps({'task_id':task_id, 'run_id':trial_b.run_id,
            'reporting_error':trace_explorer_error, 'agent_or_evaluation_status_changed':False}, indent=2))
    try:
        archive=trial_b.export()
    except Exception as exc:
        print('EXPORT HOLD:',repr(exc))
        archive=None
    agent=trial_b.agent_result or {}
    junit=(trial_b.candidate_result or {}).get('junit',{})
    raw_status = summary.get('status', 'REVIEW_REQUIRED')
    # Never classify a non-submitted or gate-bypassed patch as successfully completed.
    confirmed = (raw_status == 'PATCH_VERIFIED'
                 and bool(agent.get('patch_submitted'))
                 and int(agent.get('patch_bytes') or 0) > 0
                 and not agent.get('test_file_mutation_warning'))
    audited_status = raw_status if raw_status != 'PATCH_VERIFIED' or confirmed else 'VERIFIED_BUT_INCOMPLETE_SUBMISSION'
    batch_rows[task_id]={
        'task_id':task_id,'status':audited_status,
        'qualified':bool(trial_b.qualified),
        'independent_test_passed':junit.get('passed'),
        'independent_test_total':junit.get('tests'),
        'elapsed_seconds':experiment_elapsed_seconds,
        'model_calls':agent.get('agent_model_calls'),
        'tool_calls':agent.get('tool_calls'),
        'patch_bytes':agent.get('patch_bytes'),
        'submitted':agent.get('patch_submitted',False),
        'run_id':trial_b.run_id,'run_dir':str(trial_b.root),
        'archive':str(archive) if archive else None,
        'trace_explorer':str(trace_explorer_path) if trace_explorer_path else None,
        'trace_explorer_error':trace_explorer_error,
        'error':agent.get('agent_error') or (trial_b.holds[-1]['reason'] if trial_b.holds else None),
    }
    save_batch_results(batch_rows)
    if trace_explorer_path is not None:
        try:
            from IPython.display import display, FileLink
            display(FileLink(str(trace_explorer_path)))
        except Exception as _link_exc:
            print('HTML file available in Kaggle Output:', trace_explorer_path)
    print(f'[RESULT] {task_id}: {batch_rows[task_id]["status"]} '
          f'| passed={junit.get("passed", "NA")}/{junit.get("tests", "NA")} '
          f'| duration={batch_rows[task_id]["elapsed_seconds"]}s',flush=True)

print('\nGP02B batch register:',BATCH_CSV)

## GP02B 07 — Review cross-task results (CPU only)
Report each task separately. Infrastructure holds and unsubmitted patches are not scored as agent failures.


In [ ]:
rows=get_batch_results()
print('GP02B v2.7 SIX DEVELOPMENT TASKS | completed rows:',len([t for t in TASKS_TO_RUN if t in rows]),'/',len(TASKS_TO_RUN))
for tid in TASKS_TO_RUN:
    r=rows.get(tid)
    print(f'{tid:<15} | {r["status"]:<20} | '+
          f'{r.get("independent_test_passed")}/{r.get("independent_test_total")} verified | '+
          f'agent_calls={r.get("model_calls")} | wall_s={r.get("elapsed_seconds")}' if r else f'{tid:<15} | NOT RUN')
if rows:
    n=sum(1 for r in rows.values() if r.get('status')=='PATCH_VERIFIED')
    print(f'Observed verified patches: {n}/{len(rows)} attempted/qualified+held tasks; '
          'not a held-out benchmark estimate.')
print('Results CSV:',BATCH_CSV)
print('\nResearcher-only HTML trace explorer files:')
for _task_id, _record in sorted(rows.items()):
    _p = _record.get('trace_explorer')
    if _p:
        print(_task_id, '->', _p)
    elif _record.get('trace_explorer_error'):
        print(_task_id, '-> REPORTING HOLD:', _record['trace_explorer_error'])
    else:
        print(_task_id, '-> no HTML report recorded (older run or missing trace)')


## GP02B 08 — Evidence and hand-off to GP03

The per-task results ZIP contains full model request/response tracing, tool events, source snapshots' fingerprints and independent verifier outputs. Preserve the original v2.4 success archives separately. GP03 can build an evidence-freshness ledger from these traces without changing the GP02B model behaviour.

**Research cautions:** four shortlisted development tasks are not enough for general population claims; candidate tasks that cannot be qualified should remain infrastructure holds; do not reveal evaluator-only reference/test data to the agent or to prompts; do not claim official Docker-equivalent scoring from the subprocess sandbox.
**New in v1.5 (reporting only):** each new run’s ZIP also contains
`reports/GP02B_Gemma_Trace_Explorer_<task>_<run_id>.html`,
`reports/trace_explorer_generator.py`, and `reports/trace_explorer_manifest.json`.
The HTML is derived solely from completed evidence. Do not interpret preceding tool
results as full exact model input; no original model/evaluator files are rewritten.


In [ ]:
from IPython.display import display, FileLink
if BATCH_CSV.exists():
    display(FileLink(str(BATCH_CSV)))
if BATCH_JSON.exists():
    display(FileLink(str(BATCH_JSON)))
print('GP02B server:',server_instance.base_url)
print('GP02B is independent of prior notebook kernels.')